# System Router on Colab (spec v3)

Thin notebook: all logic lives in the repository. Use an A100 runtime for Qwen3.5-9B unquantized; on a T4 or L4 add `--set llm.quantization=4bit` (and `llm.dtype=float16` on a T4). Runs go to Google Drive; the model is downloaded to the session disk each time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/system-router/runs'
MODEL = '/content/models/Qwen3.5-9B'
SET = f'--set paths.model={MODEL} --set paths.runs={RUNS}'

In [ ]:
!git clone --recurse-submodules https://github.com/serciex/System-Router.git /content/System-Router
%cd /content/System-Router

In [ ]:
# Chrome for MiniWoB++ (Selenium downloads a matching chromedriver) and Xvfb for the desktop fallback
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get -qq update && apt-get -qq install -y ./google-chrome-stable_current_amd64.deb xvfb
!pip -q install -r requirements.txt

In [ ]:
# Stages 0 and 1: tests and adapter conformance (no model needed)
!python -m pytest -q tests
!python scripts/check_adapters.py --adapters web fallback {SET}

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen3.5-9B', local_dir=MODEL)

In [ ]:
# Stage 2: untrained loop, think-every-step baseline, fallback with narrowing vs native grounding
!python scripts/run_loop.py --episodes 20 {SET}
!python scripts/run_loop.py --episodes 20 --think-always {SET}
!python scripts/run_loop.py --episodes 20 --adapter fallback {SET}
!python scripts/run_loop.py --episodes 20 --adapter fallback --grounding native {SET}